# Geopolitical hedge — WTI crude × US strikes Iran

Data-sourced worked example, parallel to `solar_data.ipynb`.

**Episode:** a \$100k **long WTI crude futures** position (`CL=F`), hedged with the highest-volume Iran-strike contract on Polymarket.

| Input | Source |
|-------|--------|
| WTI prices & returns | [Yahoo Finance](https://finance.yahoo.com/quote/CL%3DF/history/) via `yfinance` (`CL=F`) |
| Strike contract metadata | [Polymarket Gamma API](https://gamma-api.polymarket.com) |
| Strike probability history | [Polymarket CLOB API](https://clob.polymarket.com/prices-history) |
| No-strike counterfactual | Regression of oil returns on Δstrike probability (pre-event) |
| Residual vol | Std dev of regression residuals |
| Bid-ask spread | `orderPriceMinTickSize` from Gamma × 10 ticks |

**Market picked:** *"US strikes Iran by February 28, 2026?"* — **\$89.7M** traded on this date alone (largest single-date leg of the \$529M *US strikes Iran by…?* event on Polymarket).

**Key difference from the solar example:** long oil **benefits** from a strike (prices rally). The variance-minimising hedge is therefore **short YES** (negative contract count), not long YES.

**Timing note:** the strike occurred on **Saturday Feb 28**; oil is measured from **Friday Feb 27 close → Monday Mar 2 close** (first trading day after the strike).

In [ ]:
import json
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

%matplotlib inline

GAMMA = "https://gamma-api.polymarket.com"
CLOB = "https://clob.polymarket.com"
OIL_TICKER = "CL=F"
MARKET_SLUG = (
    "us-strikes-iran-by-february-28-2026-227-967-547-688-589-491-592-418-452-924-384-915-464-672-196-157-993-596-269-535-381-391-471-256-988-997-296-225-762-973-292-827-345-182-558-215-794-879-189-761"
)
EVENT_EVE = pd.Timestamp("2026-02-27")       # last oil futures close before Feb 28 deadline
EVENT_RET_DAY = pd.Timestamp("2026-03-02")   # first oil futures close after the strike
V0 = 100_000
N_MC = 200_000
RNG = np.random.default_rng(7)


def fetch_json(url: str):
    req = urllib.request.Request(url, headers={"User-Agent": "paris-hack/hedgingresearch"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.load(resp)

## 1. Fetch Polymarket contract

In [ ]:
market = fetch_json(f"{GAMMA}/markets?slug={MARKET_SLUG}&closed=true")[0]
token_ids = json.loads(market["clobTokenIds"])
outcomes = json.loads(market["outcomes"])
yes_idx = outcomes.index("Yes")
yes_token = token_ids[yes_idx]
tick_size = float(market["orderPriceMinTickSize"])
volume_usd = float(market["volumeNum"])
yes_settled = float(json.loads(market["outcomePrices"])[yes_idx]) >= 0.99

print("Market:", market["question"])
print(f"Volume: ${volume_usd/1e6:.1f}M")
print("Resolved:", market.get("closedTime"), "→", "Yes (strike occurred)" if yes_settled else "No")
print("Min tick:", tick_size)

## 2. Fetch strike probability history (daily)

In [ ]:
history = fetch_json(
    f"{CLOB}/prices-history?market={yes_token}&interval=max&fidelity=1440"
)["history"]

poly = pd.DataFrame(history)
poly["date"] = (
    pd.to_datetime(poly["t"], unit="s", utc=True)
    .dt.tz_convert("America/New_York")
    .apply(lambda ts: ts.date())
)
strike_prob = poly.groupby("date")["p"].last().sort_index().rename("strike_prob")
strike_prob.index = pd.to_datetime(strike_prob.index)

print(f"Polymarket history: {strike_prob.index.min().date()} → {strike_prob.index.max().date()} ({len(strike_prob)} days)")
print("\nLast week before deadline:")
display(strike_prob.loc["2026-02-20":"2026-02-27"].to_frame())

## 3. Fetch WTI crude futures prices

In [ ]:
oil_close = yf.download(OIL_TICKER, start="2026-01-01", end="2026-03-10", progress=False, auto_adjust=True)["Close"]
if isinstance(oil_close, pd.DataFrame):
    oil_close = oil_close.iloc[:, 0]
oil_close.index = pd.to_datetime(oil_close.index).normalize()
oil_ret = oil_close.pct_change().rename("oil_ret")

print(f"{OIL_TICKER} around the strike:")
display(pd.DataFrame({"close": oil_close, "return": oil_ret}).loc["2026-02-24":"2026-03-04"])

## 4. Derive model parameters from data

In [ ]:
# --- [REAL] eve implied P(strike by Feb 28) from Polymarket ---
p_event = float(strike_prob.asof(EVENT_EVE))
yes_mid = p_event

# --- [REAL] realized oil move: Fri Feb 27 close → Mon Mar 2 close ---
ret_if_event = float(oil_close.loc[EVENT_RET_DAY] / oil_close.loc[EVENT_EVE] - 1)

# --- [EST] no-strike counterfactual via pre-event regression ---
panel = pd.concat([oil_ret, strike_prob], axis=1).dropna()
panel["d_strike_prob"] = panel["strike_prob"].diff()
reg_window = panel.loc[panel.index <= EVENT_EVE].dropna()

beta, intercept = np.polyfit(reg_window["d_strike_prob"], reg_window["oil_ret"], 1)
residuals = reg_window["oil_ret"] - (intercept + beta * reg_window["d_strike_prob"])
ret_if_not_reg = beta * (0.0 - p_event)

# Alternative: scale the realized move from p_eve → 1 down to p_eve → 0
beta_event = ret_if_event / (1.0 - p_event)
ret_if_not_extrap = beta_event * (0.0 - p_event)

ret_if_not = ret_if_not_reg
resid_sigma = float(residuals.std())
spread = tick_size * 10
yes_cost = yes_mid + spread

sources = pd.DataFrame([
    {"parameter": "p_event / yes_mid", "value": f"{p_event:.4f}", "source": f"Polymarket Yes close on {EVENT_EVE.date()}"},
    {"parameter": "ret_if_event (strike)", "value": f"{ret_if_event:+.2%}", "source": f"{OIL_TICKER} {EVENT_EVE.date()} → {EVENT_RET_DAY.date()} (yfinance)"},
    {"parameter": "ret_if_not (regression)", "value": f"{ret_if_not_reg:+.2%}", "source": f"β={beta:.2f} × (0 − {p_event:.2f}), n={len(reg_window)} days"},
    {"parameter": "ret_if_not (extrapolation)", "value": f"{ret_if_not_extrap:+.2%}", "source": f"Scale realized move across ({p_event:.2f} → 0)"},
    {"parameter": "resid_sigma", "value": f"{resid_sigma:.2%}", "source": "Std dev of regression residuals"},
    {"parameter": "spread", "value": f"{spread:.3f}", "source": f"10 × Gamma tick size ({tick_size})"},
])
display(sources)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(strike_prob.index, strike_prob.values, color="#8e44ad", lw=1.5)
axes[0].axvline(EVENT_EVE, color="k", ls="--", lw=0.8, label="Eve (Feb 27)")
axes[0].set_title("Polymarket: P(US strikes Iran by Feb 28)")
axes[0].set_ylabel("Probability")
axes[0].legend()

axes[1].plot(oil_close.loc["2026-02-01":"2026-03-05"], color="#2c3e50", lw=1.5)
axes[1].axvline(EVENT_EVE, color="gray", ls="--", lw=0.8, label="Eve")
axes[1].axvline(EVENT_RET_DAY, color="k", ls="--", lw=0.8, label="Post-strike")
axes[1].set_title(f"{OIL_TICKER} (WTI crude futures)")
axes[1].set_ylabel("USD/bbl")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Variance-minimising hedge

Long oil **wins** if the strike happens, so the hedge is **short YES** (negative `N`):

```
Strike:    V0×(1 + ret_strike)  + N×(1 − yes_cost)
No strike: V0×(1 + ret_no_strike) − N×yes_cost

N = V_no_strike − V_strike   (negative when oil rallies on strike)
```

In [ ]:
V_strike = V0 * (1 + ret_if_event)
V_no_strike = V0 * (1 + ret_if_not)
N = V_no_strike - V_strike          # negative → short YES
premium = N * yes_cost              # negative → premium received

wealth_strike_hedged = V_strike + N * (1 - yes_cost)
wealth_no_strike_hedged = V_no_strike - N * yes_cost

side = "SHORT" if N < 0 else "LONG"
print(f"Hedge: {side} {abs(N):,.0f} YES contracts @ ${yes_cost:.3f}")
print(f"Premium received: ${abs(premium):,.0f}" if premium < 0 else f"Premium paid: ${premium:,.0f}")
print(f"Locked wealth (two-state): ${wealth_strike_hedged:,.0f} / ${wealth_no_strike_hedged:,.0f}")

## 6. Monte Carlo with residual (basis) risk

In [ ]:
def mean_std_two_state(win_val, lose_val, p):
    m = p * win_val + (1 - p) * lose_val
    v = p * (win_val - m) ** 2 + (1 - p) * (lose_val - m) ** 2
    return m, np.sqrt(v)


def pct(x):
    return 100 * x / V0


m_unh, sd_unh = mean_std_two_state(V_strike, V_no_strike, p_event)
m_hed, sd_hed = mean_std_two_state(wealth_strike_hedged, wealth_no_strike_hedged, p_event)

event = RNG.random(N_MC) < p_event
base_ret = np.where(event, ret_if_event, ret_if_not)
noise = RNG.normal(0, resid_sigma, N_MC)
oil_sim = base_ret + noise

port_unh = V0 * (1 + oil_sim)
hedge_payoff = np.where(event, N, 0.0) - premium
port_hed = port_unh + hedge_payoff

pnl_unh = port_unh - V0
pnl_hed = port_hed - V0

var_removed = 1 - np.var(pnl_hed) / np.var(pnl_unh)
hedge_cost_usd = pnl_unh.mean() - pnl_hed.mean()
hedge_cost_bps = 1e4 * hedge_cost_usd / V0
var5_unh = np.percentile(pnl_unh, 5)
var5_hed = np.percentile(pnl_hed, 5)

## 7. Summary

In [ ]:
print("=" * 60)
print("GEOPOLITICAL HEDGE — $100k long WTI vs US-strikes-Iran contract")
print(f"Market volume: ${volume_usd/1e6:.1f}M  |  parameters from Polymarket + yfinance")
print("=" * 60)
print(f"Market-implied P(strike)       : {p_event:.1%}  [Polymarket {EVENT_EVE.date()}]")
print(f"Oil if strike / if not         : {ret_if_event:+.1%} / {ret_if_not:+.1%}")
print(f"  (alt extrapolation if not)   : {ret_if_not_extrap:+.1%}")
print(f"Contract price (buy YES)     : ${yes_cost:.3f}  (mid ${yes_mid:.3f} + {spread*100:.1f}c spread)")
print(f"Hedge size                     : {N:,.0f} contracts  ({side} YES)")
print("-" * 60)
print("TWO-STATE (strike-driven only)")
print(f"  Unhedged  E=${m_unh:,.0f}   sd=${sd_unh:,.0f}  ({pct(sd_unh):.1f}% of notional)")
print(f"  Hedged    E=${m_hed:,.0f}   sd=${sd_hed:,.0f}  (locked across states)")
print("-" * 60)
print("FULL MODEL (with residual / basis risk)")
print(f"  Unhedged P&L : mean ${pnl_unh.mean():,.0f}  sd ${pnl_unh.std():,.0f} ({pct(pnl_unh.std()):.1f}%)")
print(f"  Hedged   P&L : mean ${pnl_hed.mean():,.0f}  sd ${pnl_hed.std():,.0f} ({pct(pnl_hed.std()):.1f}%)")
print(f"  >> VARIANCE REMOVED        : {var_removed:.1%}")
print(f"  >> RESIDUAL (basis) risk   : {1 - var_removed:.1%}  (irreducible)")
print(f"  >> HEDGE COST              : ${hedge_cost_usd:,.0f}  ({hedge_cost_bps:+.0f} bps of notional)")
print("-" * 60)
print("TAIL (5% worst case)")
print(f"  Unhedged  : {pct(var5_unh):+.1f}%   (${var5_unh:,.0f})")
print(f"  Hedged    : {pct(var5_hed):+.1f}%   (${var5_hed:,.0f})")
print(f"  Drawdown cut by ~{pct(var5_hed - var5_unh):+.1f} pts in the tail")
print("=" * 60)

## 8. Chart

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))

bins = np.linspace(pct(pnl_unh.min()), pct(pnl_unh.max()), 80)
ax[0].hist(pct(pnl_unh), bins=bins, alpha=0.55, label="Unhedged", color="#c0392b")
ax[0].hist(pct(pnl_hed), bins=bins, alpha=0.75, label="Hedged", color="#16a085")
ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_title("P&L distribution (% of notional)")
ax[0].set_xlabel("Return %")
ax[0].set_ylabel("Frequency")
ax[0].legend()

labels = ["Unhedged", "Hedged"]
sds = [pct(pnl_unh.std()), pct(pnl_hed.std())]
tails = [pct(var5_unh), pct(var5_hed)]
x = np.arange(2)
ax[1].bar(x - 0.2, sds, 0.4, label="Volatility (%)", color="#2980b9")
ax[1].bar(x + 0.2, [-t for t in tails], 0.4, label="Tail loss, 5% (%)", color="#e67e22")
ax[1].set_xticks(x)
ax[1].set_xticklabels(labels)
ax[1].set_title(f"Risk: {var_removed:.0%} of variance removed for ~{abs(hedge_cost_bps):.0f} bps")
ax[1].legend()

plt.tight_layout()
plt.savefig("hedge_chart_oil_iran.png", dpi=140)
plt.show()
print("chart saved to hedge_chart_oil_iran.png")